In [1]:
from pyspark.sql import SparkSession
from pyspark.ml.feature import VectorAssembler, StandardScaler
from pyspark.ml.regression import RandomForestRegressor
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.sql.types import StructType, StructField, IntegerType, DoubleType
from pyspark.ml import Pipeline

In [2]:
# Iniciar a sessão Spark
spark = SparkSession.builder.appName("DiabetesPrediction").getOrCreate()

In [3]:
# Carregar o dataset
df = spark.read.csv("pacientes_diabetes.csv", header=True, inferSchema=True)

# Mostrar as primeiras linhas e o schema do DataFrame
print("Amostra dos dados:")
df.show(5)
print("\nEstrutura do DataFrame:")
df.printSchema()

Amostra dos dados:
+---+----------+-----+----+----------------+--------+-----------------+-------------+
| id|data_exame|idade| imc|pressao_arterial|insulina|exercicio_semanal|nivel_glicose|
+---+----------+-----+----+----------------+--------+-----------------+-------------+
|  1|2022-12-30|   77|39.5|             166|    19.1|                5|        245.6|
|  2|2020-04-16|   69|30.3|             161|    20.6|                2|        220.1|
|  3|2020-01-08|   78|27.5|             129|    13.5|                3|        203.6|
|  4|2022-05-29|   43|40.0|             133|     6.6|                3|        191.6|
|  5|2020-06-05|   67|35.1|             163|    15.6|                3|        205.0|
+---+----------+-----+----+----------------+--------+-----------------+-------------+
only showing top 5 rows

Estrutura do DataFrame:
root
 |-- id: integer (nullable = true)
 |-- data_exame: date (nullable = true)
 |-- idade: integer (nullable = true)
 |-- imc: double (nullable = true)
 |-- 

In [4]:
# Preparar os dados para o modelo
feature_columns = ["idade", "imc", "pressao_arterial", "insulina", "exercicio_semanal"]
assembler = VectorAssembler(inputCols=feature_columns, outputCol="features")
scaler = StandardScaler(inputCol="features", outputCol="scaledFeatures")

# Criar e treinar o modelo Random Forest
rf = RandomForestRegressor(featuresCol="scaledFeatures", labelCol="nivel_glicose", numTrees=100)

# Criar um pipeline
pipeline = Pipeline(stages=[assembler, scaler, rf])

# Dividir os dados em conjuntos de treino e teste
train_data, test_data = df.randomSplit([0.8, 0.2], seed=42)

# Ajustar o pipeline aos dados de treinamento
model = pipeline.fit(train_data)

# Fazer previsões no conjunto de teste


predictions = model.transform(test_data)

# Avaliar o modelo
'''
O Erro Quadrático Médio (RMSE - Root Mean Square Error, em inglês) é uma métrica comumente usada para avaliar a precisão de modelos de regressão. Ele mede a diferença média entre os valores previstos pelo modelo e os valores reais observados.
   - O RMSE é expresso na mesma unidade da variável que está sendo prevista.
   - Quanto menor o RMSE, melhor é o desempenho do modelo.
   - Um RMSE de 0 indicaria uma previsão perfeita (o que é praticamente impossível em dados do mundo real).
'''
evaluator = RegressionEvaluator(labelCol="nivel_glicose", predictionCol="prediction", metricName="rmse")
rmse = evaluator.evaluate(predictions)
print(f"\nErro Quadrático Médio (RMSE): {rmse}")


Erro Quadrático Médio (RMSE): 13.814826649154696


In [5]:
# Função para prever o nível de glicose de um novo paciente
def prever_nivel_glicose(idade, imc, pressao_arterial, insulina, exercicio_semanal):
    # Criar um DataFrame com os dados do novo paciente
    schema = StructType([
        StructField("idade", IntegerType(), True),
        StructField("imc", DoubleType(), True),
        StructField("pressao_arterial", IntegerType(), True),
        StructField("insulina", DoubleType(), True),
        StructField("exercicio_semanal", IntegerType(), True)
    ])

    novo_paciente = spark.createDataFrame([(idade, imc, pressao_arterial, insulina, exercicio_semanal)], schema)

    # Fazer a previsão
    previsao = model.transform(novo_paciente)

    return previsao.select("prediction").collect()[0][0]

In [6]:
# Exemplo de uso: prever o nível de glicose para um novo paciente
idade = 45
imc = 28.5
pressao_arterial = 130
insulina = 150.0
exercicio_semanal = 3

nivel_glicose_previsto = prever_nivel_glicose(idade, imc, pressao_arterial, insulina, exercicio_semanal)

print(f"\nPrevisão para o novo paciente:")
print(f"Idade: {idade}")
print(f"IMC: {imc}")
print(f"Pressão Arterial: {pressao_arterial}")
print(f"Insulina: {insulina}")
print(f"Exercício Semanal (horas): {exercicio_semanal}")
print(f"Nível de Glicose Previsto: {nivel_glicose_previsto:.2f}")

# Encerrar a sessão Spark
spark.stop()


Previsão para o novo paciente:
Idade: 45
IMC: 28.5
Pressão Arterial: 130
Insulina: 150.0
Exercício Semanal (horas): 3
Nível de Glicose Previsto: 193.90
